In [1]:


import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/")

In [2]:
# set or create an experiment
mlflow.set_experiment("exp6_lightgbm_optuna_advanced_hpt") 


2025/12/01 18:39:43 INFO mlflow.tracking.fluent: Experiment with name 'exp6_lightgbm_optuna_advanced_hpt' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/8', creation_time=1764594582624, experiment_id='8', last_update_time=1764594582624, lifecycle_stage='active', name='exp6_lightgbm_optuna_advanced_hpt', tags={}>

In [3]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv')

In [6]:
df['sentiment_numeric']=df.pop('sentiment_numeric')

In [9]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import optuna
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight

# -------------------------
# CONFIG
# -------------------------
N_TRIALS = 100
TFIDF_MAX_FEAT = 10000
NGRAM_RANGE = (1, 3)   # user selected B
EARLY_STOP = 50
RANDOM_STATE = 42
TEST_SIZE = 0.20
MLFLOW_RUN_NAME = "LightGBM_TFIDF_(1,3)_Optuna_Recall_100trials"

# Map target and drop missing labels/text
df['sentiment_numeric'] = df['sentiment_numeric'].map({-1: 2, 0: 0, 1: 1})


# SPLIT numeric features + text
X_numeric = df.iloc[:, 1:-1]
y = df['sentiment_numeric']

# scale numeric (with_mean=False to keep sparse compatibility)
scaler = StandardScaler(with_mean=False)
X_numeric_scaled = scaler.fit_transform(X_numeric)

# split and retain train/test indices for aligning TF-IDF
X_train_num, X_test_num, y_train, y_test, train_idx, test_idx = train_test_split(
    X_numeric_scaled, y, df.index,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# -------------------------
# TF-IDF (text) features
# -------------------------
tfidf = TfidfVectorizer(ngram_range=NGRAM_RANGE, max_features=TFIDF_MAX_FEAT)
X_train_text = tfidf.fit_transform(df.loc[train_idx, 'text_clean'])
X_test_text = tfidf.transform(df.loc[test_idx, 'text_clean'])

# combine sparse text + numeric
X_train = sp.hstack([X_train_text, sp.csr_matrix(X_train_num)], format='csr')
X_test  = sp.hstack([X_test_text,  sp.csr_matrix(X_test_num)],  format='csr')

# -------------------------
# CLASS WEIGHTS -> sample_weight
# -------------------------
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(y_train),
                                     y=y_train)
class_weight_dict = {int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights)}
sample_weights = np.array([class_weight_dict[int(lbl)] for lbl in y_train])

print("Class Weights:", class_weight_dict)

# -------------------------
# OPTUNA OBJECTIVE (maximize macro recall)
# -------------------------
def objective(trial):
    # search space tuned to avoid overfitting but still flexible
    params = {
        "objective": "multiclass",
        "num_class": len(np.unique(y)),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 2.0),
        "class_weight": class_weight_dict,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1
    }

    model = lgb.LGBMClassifier(**params)

    # use early stopping on validation set
    model.fit(
        X_train, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_test, y_test)],
        eval_metric="multi_logloss",
        callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(0)]
    )

    preds = model.predict(X_test)
    return recall_score(y_test, preds, average="macro")  # we maximize macro recall

# Create study and run
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print("\nBest Optuna params:\n", best_params)

# -------------------------
# FINAL TRAIN with best params (add fixed params)
# -------------------------
best_params.update({
    "objective": "multiclass",
    "num_class": len(np.unique(y)),
    "metric": "multi_logloss",
    "class_weight": class_weight_dict,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1
})

final_model = lgb.LGBMClassifier(**best_params)

final_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(EARLY_STOP)]
)

y_pred = final_model.predict(X_test)

# -------------------------
# REPORT + LOGGING
# -------------------------
acc = accuracy_score(y_test, y_pred)
macro_rec = recall_score(y_test, y_pred, average="macro")
report_dict = classification_report(y_test, y_pred, output_dict=True)
cm = confusion_matrix(y_test, y_pred)

print("\n=================== FINAL RESULTS ===================")
print("Accuracy:", acc)
print("Macro Recall:", macro_rec)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n")
print(cm)

# MLflow logging
with mlflow.start_run(run_name=MLFLOW_RUN_NAME):
    mlflow.log_param("algorithm", "LightGBM")
    mlflow.log_params(best_params)
    mlflow.log_param("tfidf_ngram", str(NGRAM_RANGE))
    mlflow.log_param("tfidf_max_features", TFIDF_MAX_FEAT)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("macro_recall", macro_rec)

    # log class-wise metrics
    for cls, metrics in report_dict.items():
        if isinstance(metrics, dict):
            for mname, mval in metrics.items():
                mlflow.log_metric(f"{cls}_{mname}", float(mval))

    # confusion matrix artifact
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - LightGBM (Optuna Recall)")
    plt.savefig("confusion_matrix_lgb_optuna.png")
    mlflow.log_artifact("confusion_matrix_lgb_optuna.png")
    plt.close()

    # save model + vectorizer + scaler
    joblib.dump(final_model, "lightgbm_final_optuna.pkl")
    joblib.dump(tfidf, "tfidf_vectorizer.pkl")
    joblib.dump(scaler, "numeric_scaler.pkl")
    mlflow.log_artifact("lightgbm_final_optuna.pkl")
    mlflow.log_artifact("tfidf_vectorizer.pkl")
    mlflow.log_artifact("numeric_scaler.pkl")

print("\nModel, TFIDF and scaler saved & logged. ✅")


[I 2025-12-01 18:47:52,636] A new study created in memory with name: no-name-0f1cdc17-3fb2-429c-a1e0-cbb4ef3db31c


Class Weights: {0: 0.7256444102225644, 1: 0.7350641632774342, 2: 3.8242512077294686}


  0%|          | 0/100 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's multi_logloss: 0.794141


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

[I 2025-12-01 21:20:24,930] Trial 98 finished with value: 0.6988870228859163 and parameters: {'n_estimators': 641, 'learning_rate': 0.0760499050553125, 'num_leaves': 135, 'max_depth': 15, 'feature_fraction': 0.641127278364977, 'bagging_fraction': 0.8488033846971175, 'bagging_freq': 1, 'min_child_samples': 77, 'lambda_l1': 0.7621855920918864, 'lambda_l2': 0.9093290719517193}. Best is trial 15 with value: 0.7264286830839084.
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[779]	valid_0's multi_logloss: 0.737984


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[I 2025-12-01 21:22:34,531] Trial 99 finished with value: 0.7154875492856633 and parameters: {'n_estimators': 779, 'learning_rate': 0.05998907842963967, 'num_leaves': 96, 'max_depth': 12, 'feature_fraction': 0.6055910675248382, 'bagging_fraction': 0.8104801699636373, 'bagging_freq': 0, 'min_child_samples': 5, 'lambda_l1': 0.21107844069488094, 'lambda_l2': 0.9782582877515877}. Best is trial 15 with value: 0.7264286830839084.

Best Optuna params:
 {'n_estimators': 782, 'learning_rate': 0.06359503374196453, 'num_leaves': 164, 'max_depth': 14, 'feature_fraction': 0.5023700598265448, 'bagging_fraction': 0.7825358880181615, 'bagging_freq': 3, 'min_child_samples': 5, 'lambda_l1': 0.0058664636452441035, 'lambda_l2': 0.6230400910033909}
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[782]	valid_0's multi_logloss: 0.671662


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=================== FINAL RESULTS ===================
Accuracy: 0.7093775262732417
Macro Recall: 0.7264286830839084

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.71      0.71      4546
           1       0.80      0.70      0.75      4488
           2       0.43      0.77      0.55       862

    accuracy                           0.71      9896
   macro avg       0.65      0.73      0.67      9896
weighted avg       0.73      0.71      0.72      9896


Confusion Matrix:

[[3205  725  616]
 [1079 3149  260]
 [ 140   56  666]]
🏃 View run LightGBM_TFIDF_(1,3)_Optuna_Recall_100trials at: http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/#/experiments/8/runs/9e3b642bf72f41ee812c300bc7615d8b
🧪 View experiment at: http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/#/experiments/8

Model, TFIDF and scaler saved & logged. ✅
